In [38]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128

max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-3
eval_iters = 250
n_embd = 384
n_layer = 4
n_head = 4
dropout = 0.1

cuda


In [39]:
chars = ""
with open("wizard_of_oz.txt", 'r', encoding='utf-8') as f:
        text = f.read()
        chars = sorted(list(set(text)))
        
vocab_size = len(chars)

In [40]:
# Tokenizer
# Encoder, decoder

# (map char to its idx)
string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]  # encoder: take a string, output a list of integers
decode = lambda l: ''.join(int_to_string[i] for i in l)  # decoder: take a list of integers, output a string

# encoder and decoder because Computers understand numbers, not text

# A tensor is a mathematical object representing data that has multiple dimensions
# generalization of numbers, vectors, and matrices.

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([81, 41, 57, 54,  1, 37, 67, 64, 59, 54, 52, 69,  1, 28, 70, 69, 54, 63,
        51, 54, 67, 56,  1, 54, 23, 64, 64, 60,  1, 64, 55,  1, 41, 57, 54,  1,
        44, 64, 63, 53, 54, 67, 55, 70, 61,  1, 44, 58, 75, 50, 67, 53,  1, 64,
        55,  1, 36, 75,  0,  1,  1,  1,  1,  0, 41, 57, 58, 68,  1, 54, 23, 64,
        64, 60,  1, 58, 68,  1, 55, 64, 67,  1, 69, 57, 54,  1, 70, 68, 54,  1,
        64, 55,  1, 50, 63, 74, 64, 63, 54,  1])


In [41]:
# use 80% of the dataset to train
# 20% for validation (make sure that the prediction is valid)

n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

# get batch to process parallel
# x is inputs, y is targets (predict the next token) 

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,)) # random starting point, length = batch_size
    x = torch.stack([data[i: i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

# LLM: Given the previous token, predict the next token
# create (context -> next token) training pairs

# block_size: model can look at maximum 8 previous tokens (context length)
#             block size because transformer cannot look at infinite text
# batch_size: number of training sequences processed parallel
#             speed up training

inputs:
tensor([[ 1, 69, 57,  ..., 54, 67, 10],
        [50, 69,  1,  ..., 67, 64, 69],
        [65, 69, 54,  ...,  1, 24, 64],
        ...,
        [10,  1, 40,  ..., 54, 67,  1],
        [63, 53,  1,  ..., 63, 64, 69],
        [58, 69,  1,  ...,  1, 65, 61]], device='cuda:0')
targets:
tensor([[69, 57, 54,  ..., 67, 10,  1],
        [69,  1, 72,  ..., 64, 69, 57],
        [69, 54, 67,  ..., 24, 64, 72],
        ...,
        [ 1, 40, 64,  ..., 67,  1, 54],
        [53,  1, 69,  ..., 64, 69,  1],
        [69,  1, 72,  ..., 65, 61, 54]], device='cuda:0')


In [ ]:

# Report loss helper
@torch.no_grad()  # disables gradient tracking because we don't need learning -> faster, less memory
def estimate_loss():
    out = {}
    model.eval()
    
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model.forward(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()   # mean of eval_iters loss
        
    model.train()
    return out

# temporarily turn on eval mode and then turn back to train mode for training model later
# eval mode disable dropout (random neurons are turned off to prevent overfitting), BatchNorm, training-specific behavior 
    # (Overfitting is The model memorizes the training data too much instead of learning general patterns)

In [43]:
class Head(nn.Module):
    """One head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # C: n_embd
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # COMPUTE ATTENTION SCORES
        # 1. compare each token with every other token. score is high -> more relevant
        weight = q @ k.transpose(-2, -1) * k.shape[-1]**(-0.5)          # (B,T,hs) @ (B,hs,T) -> (B,T,T). scaling part k.shape[-1]**(-0.5) equivalent / sqrt(head_size) -> prevent dot product becomes too large
        
        # 2. mask future tokens
        weight = weight.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        # [score -inf -inf -inf]
        # [score score -inf -inf]
        # [score score score -inf]
        # [score score score score]  softmax(-inf) = 0

        # 3. softmax: attention score -> probabilities
        weight = F.softmax(weight, dim=-1)
        # 4. dropout
        weight = self.dropout(weight)

        # perform weighted aggregation of the values
        v = self.value(x)
        out = weight @ v  # (B,T,T) @ (B,T,hs) -> (B,T,hs)
        # return context-aware token representations
        # out[token3] = 0.1 * value[token0]
        #               + 0.2 * value[token1]
        #               + 0.3 * value[token2]
        #               + 0.4 * value[token3]
        return out

class MultiHeadAttention(nn.Module):
    """Multi-head of self-attention in parallel"""
    # Multi-head Attention Architecture 
    # QKV (Query, Key, Value) -> n heads (scaled dot-product attention) -> concatenate results -> linear
    
    def __init__(self, num_heads, head_size):
        super().__init__() 
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):  #input: (B, T, n_embd)
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # run all heads
        out = self.proj(out)  # concatenate all heads, combine all info
        out = self.dropout(out)
        return out

    
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        """FFN with GELU activation (current standard)"""
        # Linear -> GELU -> Linear
        
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)    # dropout some neurons to avoid overfitting
        )

    def forward(self, x):
        return self.net(x)
        

class Block(nn.Module):
    """Transformer block with Pre-LayerNorm (more stable training)"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(n_embd)
        self.self_attention = MultiHeadAttention(n_embd, n_head)
        self.layer_norm2 = nn.LayerNorm(n_embd)
        self.feed_forward = FeedForward(n_embd)

    def forward(self, x):              
        # Decoder architecture
        # Multi-head attention
        # Add and Norm
        # Feed forward
        # Add and Norm
        x = x + self.self_attention(self.layer_norm1(x))
        x = x + self.feed_forward(self.layer_norm2(x))
        
        

In [44]:
class GPTLanguageModel(nn.Module):   #nn.Module is the base class for ALL neural networks in PyTorch
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)    
        # number of tokens = vocab_size, embedding dimension = vocab_size
        # current token -> probabilities of next token
        self.position_embedding_table = nn.Embedding(block_size, n_embd)  # position encoding

        self.blocks = nn.Sequential(*(Block(n_embd, n_head=n_head) for _ in range(n_layer)))  # n_layer blocks of decoders
        self.ln_f = nn.LayerNorm(n_embd)      # final layer normalization
        self.lm_head = nn.Linear(n_embd, vocab_size)  # language model head: convert hidden states into vocabulary logits

        self.apply(self._init_weights)  # initialize weights


    def _init_weights(self, module): # initialize weights as random small numbers
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        

    def forward(self, index, targets=None):     # Define the computation performed at every call
        # index: input token -> lookup the corresponding vectors in the table
        B, T = index.shape
        # logits: raw prediction scores. later convert to probabilities using softmax (in cross_entropy)

        token_embd = self.token_embedding_table(index)                    #(B, T, n_embd)
        position_embd = self.position_embedding_table(torch.arange(T, device=device))  # embedding position 0->T-1   # (T, n_embd)
        x = token_embd + position_embd    # BROADCASTING: position "duplicate" to match dimension without copying data  -> (B, T, n_embd)
        
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape          # B: batch_size, T: sequence length, C: vocab size
            logits = logits.view(B*T, C)    
            targets = targets.view(B*T)
            # flatten. cross_entropy expects shape (N, C): N = number of training examples, C = number of classes
            # batch 1 (prediction1, p2) batch2 (p3, p4) -> [p1, p2, p3, p4]
            
            loss = F.cross_entropy(logits, targets)   # computes loss: prediction i vs target i

        return logits, loss


    # generates next tokens
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)         #loss measures how wrong the model was
            # focus only on the last token after current sequence (predicted next token)
            logits = logits[:, -1, :] # becomes (B, C)

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, C)  # dim=-1 -> apply softmax across classes (C)

            # samples one token from the probability distribution
            index_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # append new token to the existing sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)

        return index



model = GPTLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

TypeError: layer_norm(): argument 'input' (position 1) must be Tensor, not NoneType

In [ ]:
# Training loop and Optimizer -> better prediction

# create a PyTorch optimizer (AdamW is a popular optimizer for transformer, similar to gradient descent but smarter)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    # Report loss
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # predict then evaluate the loss
    logits, loss = model.forward(xb, yb)

    optimizer.zero_grad(set_to_none=True)  # 1. clear old gradients before loss.backward() (gradient is the slope/derivative)
    loss.backward()                        # 2. compute gradient for each param (backpropagation)
    optimizer.step()                       # 3. update the model parameters (weights) using the gradients (~ w new = w - learning_rate * gradient)

print(loss.item())


# Training Loop
# for many iterations:
#     get batch of data
#     make predictions
#     compute loss (how wrong the model is)
#     compute gradients
#     update weights

In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

In [ ]:
# ----------------------- Gradient Descent -----------------------------
#gradient (rate of change) descent (slope)

# Maximization and minimization
# what is the x value, find min or max output of f(x)
# Min or max point has Derivative of 0

# Walk toward that min, max point
# x next = x now - learning_rate * derivative * x now
# learning_rate is very small step

# https://docs.google.com/presentation/d/1nHesO6x48OHMN-XWljFAwnFa1MNCRKtMmj9pEVlNLfs/edit?usp=sharing


# ---------------------- Cross Entropy ------------------------------------
# loss function to measure the difference between two probability distributions
# between true labels (P) and the model's predictions (Q)
# Smaller loss: better predictions

# https://docs.google.com/presentation/d/1ZKqgPiYKPKGgjvbnf03aWKxBtSm4NRmO6qHzlZWkjbA/edit?usp=sharing

In [ ]:
# ---------------- Normalization ---------------------------
# Rescaling data into a more stable and consistent range so models can learn better
    
# | Type                  | Used For                          |
# | --------------------- | --------------------------------- |
# | Input normalization   | Scale raw data                    |
# | Standardization       | Center around mean                |
# | BatchNorm             | Normalize hidden activations      |
# | LayerNorm             | Normalize Transformer activations |
# | Min-Max normalization | Scale into fixed range            |


In [ ]:
# Activation functions: ReLU, Sigmoid, tanh
# introduces nonlinearity -> network can approximate extremely complex functions

# ReLU(x) = max(0,x)    better

# Sigmoid: S-shape, out range [0, 1]
# tanh: S-shape, out range [-1, 1]
# zero-centered outputs, better than sigmoid
# both Sigmoid and tanh causes vanishing gradients (large input -> out is either -1 or 1]

In [ ]:
# --------------- Transformer Architecture --------------------------
# Attention is all you need (https://arxiv.org/pdf/1706.03762)

In [ ]:
# -------------- Standard Deviation ---------------------------------
# measures the amount of variation or dispersion of a dataset relative to its mean

In [ ]:
# ModuleList vs Sequential
# ModuleList (Multi heads)
# can operate independently and in parallel

# Sequential (Decoder)
# Have to wait for result before the next decoder operates